In [27]:
%load_ext autotime


The autotime extension is already loaded. To reload it, use:
  %reload_ext autotime
time: 217 µs (started: 2026-09-05 22:03:58 +05:30)


In [28]:
#import torch
#from transformers import AutoTokenizer, AutoModelForCausalLM

#model_name = "distilgpt2"  # small causal language model from Hugging Face

#tokenizer = AutoTokenizer.from_pretrained(model_name)
#model = AutoModelForCausalLM.from_pretrained(model_name)

#prompt = "Donald Trump is"
#inputs = tokenizer(prompt, return_tensors="pt")

#with torch.no_grad():
#    output_ids = model.generate(
#        **inputs,
#        max_new_tokens=30,    # how long a continuation
#        do_sample=True,       # sample (don't loop greedily)
#        temperature=0.7,      # how random
#        pad_token_id=tokenizer.eos_token_id,
#    )

#generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

#print("Prompt:")
#print(prompt)
#print("\nGenerated continuation:")
#print(generated_text)

time: 236 µs (started: 2026-09-05 22:03:58 +05:30)


In [29]:
# ── 1. Environment check ─────────────────────────────────────────────────
# This cell verifies:
#   1. Required packages are installed.
#   2. PyTorch can see Apple Silicon GPU via MPS, or CUDA, or CPU.
#   3. We choose a reasonable compute dtype.

import importlib.util
import warnings

warnings.filterwarnings("ignore")

required = ["torch", "transformers", "peft", "datasets", "accelerate"]
missing = [p for p in required if importlib.util.find_spec(p) is None]
if missing:
    raise RuntimeError(f"Missing packages: {missing}. Install them first.")

import torch

# Pick the best available device.
# MPS = Apple Metal Performance Shaders backend.
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

# Prefer bfloat16 where practical. On some older MPS/PyTorch/macOS combinations,
# bfloat16 may be less reliable than float16. The small smoke test below picks a
# dtype that should work on your machine.
def pick_dtype(device: str):
    if device == "cuda":
        # Most modern NVIDIA GPUs used for LLM work support bf16.
        if torch.cuda.is_bf16_supported():
            return torch.bfloat16
        return torch.float16

    if device == "mps":
        # MPS support varies by PyTorch/macOS version. Try bf16, fall back to fp16.
        try:
            _ = torch.ones(1, device="mps", dtype=torch.bfloat16) + 1
            return torch.bfloat16
        except Exception:
            return torch.float16

    # CPU can use fp32 reliably. Training will be slow, but it avoids dtype surprises.
    return torch.float32

DTYPE = pick_dtype(DEVICE)

print(f"PyTorch      : {torch.__version__}")
print(f"Device       : {DEVICE}")
print(f"Compute dtype: {DTYPE}")

PyTorch      : 2.10.0
Device       : mps
Compute dtype: torch.bfloat16
time: 2.12 ms (started: 2026-09-05 22:03:58 +05:30)


In [30]:
# ── 2. Load tokenizer and base model ─────────────────────────────────────

from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# The tokenizer converts text <-> token IDs.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Llama-family tokenizers often do not define a dedicated pad token.
# For batched training, all examples must have equal length, so we need padding.
# A common convention is to reuse EOS as PAD.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model weights directly in the chosen dtype.
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=DTYPE,
).to(DEVICE)

# KV cache is useful for generation/inference, not training.
# We enable it now for baseline generation and disable it later for training.
base_model.config.use_cache = True

# TinyLlama ships a built-in max_length=2048. Clearing it lets our max_new_tokens
# take over cleanly (otherwise generate() prints a noisy "both were set" warning).
base_model.generation_config.max_length = None

total_params = sum(p.numel() for p in base_model.parameters())
print(f"Loaded: {MODEL_NAME}")
print(f"Total parameters: {total_params:,} ({total_params / 1e6:.1f}M)")
print(f"Approx weight memory at current dtype: {total_params * torch.tensor([], dtype=DTYPE).element_size() / 1e9:.2f} GB")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loaded: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Total parameters: 1,100,048,384 (1100.0M)
Approx weight memory at current dtype: 2.20 GB
time: 4.42 s (started: 2026-09-05 22:03:58 +05:30)


In [31]:
#"Tell me what's list comprehension"

#"role": "user", "content": "Tell me what's list comprehension"
#"role": "assistant", "content": "List comprehension is a concise way to create lists in Python. It allows you to generate a new list by applying an expression to each item in an existing iterable (like a list, tuple, or string) and optionally filtering items based on a condition. The basic syntax is:\n\n```python\nnew_list = [expression for item in iterable if condition]\n```\n\nFor example, to create a list of squares of even numbers from 0 to 9, you can use:\n\n```python\nsquares_of_even = [x**2 for x in range(10) if x % 2 == 0]\n```\n\nThis will result in `squares_of_even` being `[0, 4, 16, 36, 64]`."

time: 291 µs (started: 2026-09-05 22:04:03 +05:30)


## How do we know whether to use `apply_chat_template`?

Use it when you are working with a **chat** or **instruct** model.

Model names often contain words like:

```text
Chat
Instruct
Assistant
SFT
RLHF
DPO
```

Different chat models use different special-token formats. For example, one model may expect:

```text
<|user|>
...
<|assistant|>
```

while another may expect:

```text
[INST] ... [/INST]
```

So do **not** manually invent the prompt format unless you have to. First inspect the tokenizer's chat template.

In [32]:
# ── Inspect the model's chat template ─────────────────────────────────────
# This is useful when learning a new chat model.
# If this prints a Jinja-style template, the model/tokenizer knows how to format chat messages.

print(tokenizer.chat_template[:1000] if tokenizer.chat_template else "No chat template found.")

{% for message in messages %}
{% if message['role'] == 'user' %}
{{ '<|user|>
' + message['content'] + eos_token }}
{% elif message['role'] == 'system' %}
{{ '<|system|>
' + message['content'] + eos_token }}
{% elif message['role'] == 'assistant' %}
{{ '<|assistant|>
'  + message['content'] + eos_token }}
{% endif %}
{% if loop.last and add_generation_prompt %}
{{ '<|assistant|>' }}
{% endif %}
{% endfor %}
time: 251 µs (started: 2026-09-05 22:04:03 +05:30)


In [35]:
# ── 3. Baseline generation helper ────────────────────────────────────────

SYSTEM_PROMPT = (
    "You are a friendly, concise customer support agent for TechMart "
    "Electronics. Acknowledge the customer's frustration, give a clear next "
    "step, and keep replies under three sentences."
)


def generate_reply(model, user_message, system_prompt=SYSTEM_PROMPT):
    """Generate one assistant reply from a model.

    Key ideas:
    - We build a list of chat messages.
    - `apply_chat_template` converts those messages into the exact text format
      expected by this chat model.
    - `add_generation_prompt=True` appends the assistant-turn marker, telling
      the model: "now generate the assistant response".
    - `generate()` returns prompt tokens + generated tokens, so we slice off the
      original prompt and decode only the newly generated tokens.
    """

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},
    ]

    # Step 1: format chat messages into a single prompt string.
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # Step 2: tokenize the prompt string into tensor IDs.
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    model.eval()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=120
        )

    # Shape explanation:
    # output_ids has shape [batch_size, total_sequence_length].
    # Here batch_size is 1, so output_ids[0] selects the only generated sequence.
    #
    # `inputs["input_ids"].shape[1]` is the number of prompt tokens.
    # `output_ids[0, prompt_len:]` removes the prompt and keeps only new tokens.
    prompt_len = inputs["input_ids"].shape[1]
    new_tokens = output_ids[0, prompt_len:]

    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

time: 628 µs (started: 2026-09-05 22:04:42 +05:30)


In [36]:
# ── Run baseline prompts before fine-tuning ──────────────────────────────

test_prompts = [
    "My order #4521 hasn't arrived after 2 weeks.",
    "I want a refund for my broken headphones.",
    "Your app keeps crashing on my phone.",
]

print("=" * 80)
print("  BASE MODEL — replies before fine-tuning")
print("=" * 80)

for q in test_prompts:
    print(f"\nCustomer: {q}")
    print(f"Agent   : {generate_reply(base_model, q)}")

  BASE MODEL — replies before fine-tuning

Customer: My order #4521 hasn't arrived after 2 weeks.
Agent   : I don't have access to your specific order number, but I can provide you with a general response. Please provide me with your order number so that I can check on your order status. If your order has not yet arrived, please provide me with the date of the order, the product you ordered, and the shipping address. I will then check with our fulfillment center to see if your order has been shipped and when it will arrive. If you have any further questions or concerns, please do not hesitate to reach out to our customer support team. We are always here

Customer: I want a refund for my broken headphones.
Agent   : Sure, here's a sample response to a customer complaint about a broken headphone:

Customer: Hi, I'm sorry to hear about your broken headphones. Can you please provide me with a refund for the product?

Agent: Of course, I'd be happy to provide you with a refund for your brok

In [38]:
# ── 4. Small supervised chat dataset ─────────────────────────────────────
# Each tuple is: (customer_message, ideal_assistant_reply)


support_examples = [
    ("My order hasn't arrived yet.",
     "Arr, sorry for the delay, matey — that be vexin'. Could ye share yer order number so I can pull up the trackin' right now?"),
    ("I want a refund for my broken headphones.",
     "That not be the voyage we want for ye. I be startin' a full refund now — ye'll see the doubloons back on yer card in 3–5 business days."),
    ("How do I reset my password?",
     "Happy to help, matey. Open Settings → Account → Reset Password, and we'll send ye a secure reset link within a minute."),
    ("Your app keeps crashing on my phone.",
     "Sorry for the squall. Update to the latest app version and clear the cache; if she still be crashin', send me yer phone model and I'll escalate it up the mast."),
    ("I was charged twice for the same item.",
     "Apologies — that be plain wrong. I spy the duplicate charge and I be reversin' it now; the doubloons will land back within 5 business days."),
    ("Can I change my delivery address?",
     "Aye, as long as the order hasn't set sail. Send me the new address and I'll update it on the spot."),
    ("The product I received is the wrong color.",
     "Sorry for the mix-up, matey! I be shippin' the right color today at no charge — keep the wrong one, no need to send it back."),
    ("I need to cancel my subscription.",
     "No worries, I be cancellin' it now. Yer access stays afloat until the end of the current billing period."),
    ("The website won't accept my coupon code.",
     "Let's set that right. Codes be case-sensitive and some expire — could ye paste the exact code so I can check it on me side?"),
    ("I never received my confirmation email.",
     "Sorry 'bout that, matey. Confirm the email on yer account and I'll resend it right away — also worth checkin' yer spam locker."),
    ("My package arrived damaged.",
     "That be a cryin' shame — I be sendin' a replacement at no charge today. Ye can keep or recycle the damaged one, no return needed."),
    ("How long does shipping usually take?",
     "Standard shippin' be 5–7 business days. We also offer express (2–3 days) and overnight if ye need it sooner, matey."),
    ("Do you ship internationally?",
     "Aye, we sail to 40+ countries. Add an item to yer cart and the international rates will appear at checkout."),
    ("My order shows delivered but I never got it.",
     "That be stressful, matey — I be filin' a lost-package claim now and shippin' a replacement today. Could ye confirm the delivery address on file?"),
    ("Is the warranty transferable if I gift this?",
     "Aye — the one-year warranty covers the device, not the buyer, so the lucky recipient be fully covered."),
]

print(f"Dataset size: {len(support_examples)} examples")

Dataset size: 15 examples
time: 740 µs (started: 2026-09-05 22:06:02 +05:30)


In [ ]:
# ── Mask #1: the ATTENTION mask — padding must not soak up attention ──────
import torch

# Imagine ONE query token scoring how relevant each of 5 key positions is.
# The last two positions are PADDING (only there to make the batch rectangular),
# and — awkwardly — they happen to have high raw scores.
scores    = torch.tensor([2.0, 1.0, 0.5, 3.0, 2.5])   # raw attention scores (higher = more relevant)
attn_mask = torch.tensor([1,   1,   1,   0,   0])      # 1 = real token, 0 = padding

def rounded(t):
    return [round(v, 3) for v in t.tolist()]

# WITHOUT masking, softmax happily hands most of the attention to the padding slots (wrong!).
print("attention if we IGNORE the mask :", rounded(torch.softmax(scores, dim=-1)))

# WITH masking, we set padding scores to -inf *before* softmax, so they get exactly 0 weight.
masked = scores.masked_fill(attn_mask == 0, float("-inf"))
print("attention USING the mask        :", rounded(torch.softmax(masked, dim=-1)))

print("\n-> without the mask, 73% of attention leaks onto the 2 padding tokens;")
print("   with the mask, padding gets 0.0 and the real tokens share 100%.")

In [39]:
# ── 5. Dataset with response-only masking ────────────────────────────────

from torch.utils.data import Dataset, DataLoader

MAX_LENGTH = 256


class SupportChatDataset(Dataset):
    """Convert (user, assistant) pairs into tensors for chat SFT.

    For every example we produce:
    - input_ids: full chat sequence tokens
    - attention_mask: 1 for real tokens, 0 for padding
    - labels: same as input_ids only for assistant-response tokens;
              -100 everywhere else

    The model sees the full conversation, but loss is computed only on the
    assistant reply.
    """

    def __init__(self, pairs, tokenizer, system_prompt, max_length=MAX_LENGTH):
        self.items = []

        for user_msg, assistant_msg in pairs:
            # A. Build the prefix used at inference time:
            #    system + user + assistant header.
            #
            # We need this prefix length so we know where the assistant answer
            # begins in the full training sequence.
            prefix_messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_msg},
            ]
            prefix_text = tokenizer.apply_chat_template(
                prefix_messages,
                tokenize=False,
                add_generation_prompt=True,
            )

            # B. Build the full training text:
            #    system + user + assistant answer.
            full_messages = prefix_messages + [
                {"role": "assistant", "content": assistant_msg},
            ]
            full_text = tokenizer.apply_chat_template(
                full_messages,
                tokenize=False,
                add_generation_prompt=False,
            )

            # C. Tokenize full text with fixed padding.
            #    This makes every item exactly max_length tokens.
            full_enc = tokenizer(
                full_text,
                truncation=True,
                max_length=max_length,
                padding="max_length",
                return_tensors="pt",
            )

            # D. Tokenize prefix separately to find the assistant-answer boundary.
            #    `add_special_tokens=False` is important because the chat template
            #    has already inserted special tokens.
            prefix_enc = tokenizer(
                prefix_text,
                truncation=True,
                max_length=max_length,
                add_special_tokens=False,
                return_tensors="pt",
            )

            # Remove the batch dimension from each single example.
            # Before squeeze: [1, max_length]
            # After squeeze : [max_length]
            input_ids = full_enc["input_ids"].squeeze(0)
            attention_mask = full_enc["attention_mask"].squeeze(0)
            prefix_len = prefix_enc["input_ids"].shape[1]

            # E. Create labels.
            #    Start from a copy of input_ids, then mask everything except the
            #    assistant reply.
            labels = input_ids.clone()
            labels[:prefix_len] = -100          # ignore system + user + assistant header
            labels[attention_mask == 0] = -100  # ignore padding

            self.items.append(
                {
                    "input_ids": input_ids,
                    "attention_mask": attention_mask,
                    "labels": labels,
                }
            )

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        return self.items[idx]


dataset = SupportChatDataset(support_examples, tokenizer, SYSTEM_PROMPT)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

# Sanity check: the number of trainable-label tokens should roughly match the
# assistant reply length, not the whole sequence length.
first = dataset[0]
n_train = (first["labels"] != -100).sum().item()
n_total = first["attention_mask"].sum().item()
print(f"Example 0: {n_total} non-pad tokens, {n_train} of them contribute to loss")
print(f"           ({n_train / n_total:.0%} of visible tokens contribute to loss)")

Example 0: 110 non-pad tokens, 38 of them contribute to loss
           (35% of visible tokens contribute to loss)
time: 7.19 ms (started: 2026-09-05 22:17:21 +05:30)


In [ ]:
# ── 6. Configure and attach LoRA ─────────────────────────────────────────

from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16, # weight = lora_alpha / r = 2
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

# This freezes the base model and inserts trainable LoRA adapters into the
# selected target modules.
model = get_peft_model(base_model, lora_config)

# Disable KV cache during training. It is an inference optimization, not needed
# for full-sequence training with gradients.
model.config.use_cache = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print("=" * 60)
print("  LoRA wraps TinyLlama")
print("=" * 60)
print(f"  Total parameters     : {total:>14,}")
print(f"  Trainable LoRA params: {trainable:>14,}")
print(f"  Frozen base params   : {total - trainable:>14,}")
print(f"  % trainable          : {100 * trainable / total:>13.4f}%")

  LoRA wraps TinyLlama
  Total parameters     :  1,102,301,184
  Trainable LoRA params:      2,252,800
  Frozen base params   :  1,100,048,384
  % trainable          :        0.2044%
time: 503 ms (started: 2026-09-05 22:18:33 +05:30)


In [41]:
# ── 7. Train LoRA adapter ────────────────────────────────────────────────

import time
from torch.optim import AdamW

NUM_EPOCHS = 5
LEARNING_RATE = 2e-4

optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
)

model.train()
loss_history = []

print(f"Training on {DEVICE} for {NUM_EPOCHS} epochs...")
print("-" * 60)
start = time.time()

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0.0

    for batch in loader:
        # Move input_ids, attention_mask, labels to the same device as model.
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        optimizer.zero_grad()

        # Because labels are passed, the model returns outputs.loss.
        # That loss is computed only where labels != -100.
        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(loader)
    loss_history.append(avg_loss)

    bar_len = 30
    filled = int(bar_len * (epoch + 1) / NUM_EPOCHS)
    bar = "█" * filled + "░" * (bar_len - filled)
    print(f"  Epoch {epoch + 1}/{NUM_EPOCHS} |{bar}| loss = {avg_loss:.4f}")

elapsed = time.time() - start
print("-" * 60)
print(f"Done in {elapsed:.1f}s ({elapsed / NUM_EPOCHS:.1f}s per epoch)")

Training on mps for 5 epochs...
------------------------------------------------------------
  Epoch 1/5 |██████░░░░░░░░░░░░░░░░░░░░░░░░| loss = 3.7008
  Epoch 2/5 |████████████░░░░░░░░░░░░░░░░░░| loss = 2.5926
  Epoch 3/5 |██████████████████░░░░░░░░░░░░| loss = 2.1467
  Epoch 4/5 |████████████████████████░░░░░░| loss = 1.6724
  Epoch 5/5 |██████████████████████████████| loss = 1.2287
------------------------------------------------------------
Done in 20.1s (4.0s per epoch)
time: 20.1 s (started: 2026-09-05 22:19:33 +05:30)


In [42]:
# ── 8. Side-by-side generation ──────────────────────────────────────────

# Re-enable the KV cache for fast generation (training turned it off).
model.config.use_cache = True
base_model.config.use_cache = True

eval_prompts = [
    "My order #4521 hasn't arrived after 2 weeks.",
    "I want a refund.",
    "Your app keeps crashing.",
    "My laptop screen is flickering since the last update.",
]

print("=" * 90)
print("  BASE  vs  FINE-TUNED")
print("=" * 90)

for q in eval_prompts:
    print(f"\nCustomer    : {q}")

    # Adapter disabled: base TinyLlama behavior.
    with model.disable_adapter():
        base_reply = generate_reply(model, q)

    # Adapter enabled: TechMart LoRA behavior.
    ft_reply = generate_reply(model, q)

    print(f"  base      : {base_reply}")
    print(f"  fine-tuned: {ft_reply}")

  BASE  vs  FINE-TUNED

Customer    : My order #4521 hasn't arrived after 2 weeks.
  base      : I don't have access to your specific order number, but I can provide you with a general response. Please provide me with your order number so that I can check on your order status. If your order has not yet arrived, please provide me with the date of the order, the product you ordered, and the shipping address. I will then check with our fulfillment center to see if your order has been shipped and when it will arrive. If you have any further questions or concerns, please do not hesitate to reach out to our customer support team. We are always here
  fine-tuned: That be a shame — I be sending a replacement today and the original be back on its way now.

Customer    : I want a refund.
  base      : Customer: "I want a refund for the product I purchased from TechMart Electronics."

Reply: "We understand your concern and would like to offer you a refund for the product. Please provide us with y

In [43]:
import os

ADAPTER_DIR = "./tinyllama-techmart-lora"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

('./tinyllama-techmart-lora/tokenizer_config.json',
 './tinyllama-techmart-lora/chat_template.jinja',
 './tinyllama-techmart-lora/tokenizer.json')

time: 921 ms (started: 2026-09-05 22:21:12 +05:30)
